In [ ]:

"""
FSAE Rules → Exact, Non-overlapping Rule Texts → JSONL Dataset for Fine-tuning

"""

import json
import re
import sys
from collections import OrderedDict
from pathlib import Path
from typing import Dict, List, Tuple, Optional

PDF_PATH = Path("dataset/docs/FSAE_Rules_2024_V1.pdf")

OUTPUT_DIR = PDF_PATH.parent
RULES_INDEX_JSON = OUTPUT_DIR / "rules_index.json"
DATASET_JSONL    = OUTPUT_DIR / "rules_qa_dataset.jsonl"
RULE_IDS_TXT     = OUTPUT_DIR / "rule_ids_only.txt"

PRINT_IDS_ONLY = False

USER_INPUT_TEMPLATE = (
    "We are a student engineering team designing a vehicle for the FSAE competition. "
    "Attached is the FSAE rules document. What does rule {rule_id} state exactly? "
    "Answer with only the text of the rule and no other words."
)

RULE_ID_RE = re.compile(
    r'^\s*('
    r'[A-Z]{1,3}\.'
    r'(?:\d+(?:\.\d+)*)'
    r'(?:[a-z]|(?:\.[a-z]+))?'
    r')\b'
)

def extract_text_pdfminer(pdf_path: Path) -> Optional[str]:
    try:
        from pdfminer.high_level import extract_text
        return extract_text(str(pdf_path))
    except Exception:
        return None

def extract_text_pypdf(pdf_path: Path) -> Optional[str]:
    try:
        from pypdf import PdfReader
        reader = PdfReader(str(pdf_path))
        pages = [(p.extract_text() or "") for p in reader.pages]
        return "\n".join(pages)
    except Exception:
        return None

def read_pdf_text(pdf_path: Path) -> str:
    text = extract_text_pdfminer(pdf_path)
    if text is None:
        text = extract_text_pypdf(pdf_path)
    if text is None:
        raise RuntimeError("Failed to extract text from PDF with pdfminer and pypdf.")
    return text

def normalize_spaces(s: str) -> str:
    """Preserve newlines; compact intra-line whitespace; collapse multiple blank lines."""
    lines = [re.sub(r'[ \t]+', ' ', ln).strip() for ln in s.splitlines()]
    out: List[str] = []
    prev_blank = False
    for ln in lines:
        blank = (ln == "")
        if blank and prev_blank:
            continue
        out.append(ln)
        prev_blank = blank
    return "\n".join(out).strip()

def leading_rule_id_strip(text: str, rule_id: str) -> str:
    """
    Remove leading 'rule_id' if present at the start of the text, along with a nearby separator.
    Example:
      "AA.1.1.1 - The car must ..." → "The car must ..."
    """
    pat = re.compile(r'^\s*' + re.escape(rule_id) + r'\s*[:\-\)\.]?\s*', re.IGNORECASE)
    return pat.sub("", text, count=1)

def count_numeric_levels(rule_id: str) -> int:
    """
    Count numeric levels after the prefix letters.
    e.g., 'AA.1.1' → 2, 'F.3.5.3b' → 3, 'T.7.6.1.a' → 3
    """
    parts = rule_id.split(".")
    numeric_parts = parts[1:]
    lvl = 0
    for p in numeric_parts:
        m = re.match(r'^(\d+)', p)
        if m:
            lvl += 1
        else:
            break
    return lvl

def natural_key(rule_id: str) -> Tuple:
    """
    Natural sort key so that .10 > .3 numerically, with trailing letters as tiebreakers.
    """
    m = re.match(r'^([A-Z]{1,3})\.([0-9a-zA-Z\.\-]+)$', rule_id)
    if not m:
        return (rule_id,)
    letters = m.group(1)
    rest = m.group(2)
    parts = rest.split(".")
    key = [letters]
    for p in parts:
        d = re.match(r'^(\d+)([a-zA-Z]*)$', p)
        if d:
            key.append(int(d.group(1)))
            if d.group(2):
                key.append(d.group(2))
        else:
            key.append(p)
    return tuple(key)

def parse_rules(text: str) -> Dict[str, str]:
    """
    Single-pass stack parser that assigns content exclusively to the deepest open rule.
    - When encountering a new rule at level L, pop any rules with level >= L.
    - Start the new rule, pushing (id, level) on the stack.
    - Non-rule lines are appended only to the deepest rule.
    Returns OrderedDict[rule_id -> cleaned_text_without_leading_id]
    """
    lines = text.splitlines()
    rules_lines: "OrderedDict[str, List[str]]" = OrderedDict()
    stack: List[Tuple[str, int]] = []

    def append_to_current(line: str):
        if stack:
            rid = stack[-1][0]
            rules_lines.setdefault(rid, []).append(line)

    for raw in lines:
        line = raw.rstrip()
        m = RULE_ID_RE.match(line)
        if m:
            rid = m.group(1).strip()
            lvl = count_numeric_levels(rid)
            while stack and stack[-1][1] >= lvl:
                stack.pop()
            rules_lines.setdefault(rid, [])
            stack.append((rid, lvl))
            after = line[m.end():].strip()
            if after:
                rules_lines[rid].append(after)
        else:
            if line.strip() != "":
                append_to_current(line)
            else:
                append_to_current("")

    final: "OrderedDict[str, str]" = OrderedDict()
    for rid, segs in rules_lines.items():
        raw_txt = "\n".join(segs).strip()
        raw_txt = normalize_spaces(raw_txt)
        cleaned = leading_rule_id_strip(raw_txt, rid).strip()
        final[rid] = cleaned
    return final

def write_rules_index(rules: Dict[str, str], path: Path) -> None:
    with path.open("w", encoding="utf-8") as f:
        json.dump(rules, f, ensure_ascii=False, indent=2)

def write_dataset_jsonl(rules: Dict[str, str], path: Path) -> None:
    """
    Writes JSONL with ONLY instruction/input/output (no rule_id field).
    Also writes a sidecar index mapping row_number -> rule_id for provenance.
    """
    sidecar = path.with_name(path.stem + "_index.tsv")
    idx_rows = []

    with path.open("w", encoding="utf-8") as f:
        for i, (rid, txt) in enumerate(rules.items()):
            obj = {
                "instruction": "",
                "input": (
                    "What does rule {rule_id} is about? "
                    "return only the text of the rule"
                ).format(rule_id=rid),
                "output": txt
            }
            f.write(json.dumps(obj, ensure_ascii=False) + "\n")
            idx_rows.append(f"{i}\t{rid}")

    with sidecar.open("w", encoding="utf-8") as sf:
        sf.write("row\trule_id\n")
        sf.write("\n".join(idx_rows))

def main():
    if not PDF_PATH.exists():
        raise FileNotFoundError(f"PDF not found: {PDF_PATH}")

    print(f"[INFO] Reading PDF: {PDF_PATH}", file=sys.stderr)
    raw_text = read_pdf_text(PDF_PATH)
    raw_text = raw_text.replace("\r", "")

    print("[INFO] Parsing rules…", file=sys.stderr)
    rules_map = parse_rules(raw_text)

    items = sorted(rules_map.items(), key=lambda kv: natural_key(kv[0]))
    rules_map_sorted = OrderedDict(items)
    rule_ids = list(rules_map_sorted.keys())

    if PRINT_IDS_ONLY:
        print(",".join(rule_ids))
        return

    with RULE_IDS_TXT.open("w", encoding="utf-8") as f:
        f.write(",".join(rule_ids))

    print(f"[INFO] Found {len(rule_ids)} rule IDs.", file=sys.stderr)
    print(f"[INFO] Writing index → {RULES_INDEX_JSON}", file=sys.stderr)
    write_rules_index(rules_map_sorted, RULES_INDEX_JSON)

    print(f"[INFO] Writing dataset → {DATASET_JSONL}", file=sys.stderr)
    write_dataset_jsonl(rules_map_sorted, DATASET_JSONL)

    print("[DONE]")
    print(f" - Rule IDs (comma-separated): {RULE_IDS_TXT}")
    print(f" - Rule index (ID → exact text): {RULES_INDEX_JSON}")
    print(f" - QA dataset (JSONL): {DATASET_JSONL}")

if __name__ == "__main__":
    main()


[INFO] Reading PDF: /home/knk22002/Downloads/Hackathon_ASME/FineTurningModel/FSAE_Rules_2024_V1.pdf


[DONE]
 - Rule IDs (comma-separated): /home/knk22002/Downloads/Hackathon_ASME/FineTurningModel/rule_ids_only.txt
 - Rule index (ID → exact text): /home/knk22002/Downloads/Hackathon_ASME/FineTurningModel/rules_index.json
 - QA dataset (JSONL): /home/knk22002/Downloads/Hackathon_ASME/FineTurningModel/rules_qa_dataset.jsonl


[INFO] Parsing rules…
[INFO] Found 1759 rule IDs.
[INFO] Writing index → /home/knk22002/Downloads/Hackathon_ASME/FineTurningModel/rules_index.json
[INFO] Writing dataset → /home/knk22002/Downloads/Hackathon_ASME/FineTurningModel/rules_qa_dataset.jsonl
